In [3]:
### Importation des librairies ###


import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import rasterio
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path
from fonction import *
import seaborn as sns
import gc
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics
import xgboost as xgb
import time

In [4]:
### Importation de la base ###


# Importation
gdf = gpd.read_file('data/processed/jointure_meteo_swi_argile_nearest.gpkg')

# Affichage
display(gdf)

,NUMERO,LAMBX,LAMBY,DATE,SWI_UNIF_MENS,PRENEI,PRELIQ,T,FF,Q,...,WG_RACINE,WGI_RACINE,TINF_H,TSUP_H,YEAR,ALEA,NIVEAU,DPT,dist_to_argile,geometry
0,3827,773980,6713410,1960-12-01,0.982,5.7,59.2,1.941935,3.132258,4.049000,...,0.370645,0.003516,-5.7,12.0,1960,Moyen,2.0,89,0.000000,POINT (723999.802 2280998.23)
1,3826,765986,6713478,1960-12-01,0.977,4.8,58.9,2.216129,3.254839,4.104935,...,0.351065,0.002323,-5.4,12.3,1960,Moyen,2.0,89,0.000000,POINT (715999.436 2280998.385)
2,3825,757993,6713546,1960-12-01,0.973,4.4,58.9,2.338710,3.300000,4.130161,...,0.334355,0.001806,-5.2,12.4,1960,Moyen,2.0,89,0.000000,POINT (708000.072 2280998.598)
3,3824,749999,6713614,1960-12-01,0.973,4.4,58.9,2.338710,3.300000,4.130161,...,0.324452,0.001806,-5.2,12.4,1960,Faible,1.0,89,0.000000,POINT (699999.708 2280998.852)
4,5323,717146,6609971,1960-09-01,0.449,0.0,73.6,13.356667,1.530000,8.058967,...,0.263533,0.000000,2.3,24.0,1960,Moyen,2.0,3,0.000000,POINT (667998.94 2176998.552)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7005175,7245,363949,6429068,2024-08-01,0.124,0.0,37.5,20.451613,1.880645,11.601839,...,0.144290,0.000000,9.6,37.0,2024,Aucun,0.0,33,24729.333576,POINT (315997.597 1992998.292)
7005176,7245,363949,6429068,2024-03-01,1.042,0.0,160.8,10.851613,2.603226,6.793806,...,0.222710,0.000000,2.0,22.7,2024,Aucun,0.0,33,24729.333576,POINT (315997.597 1992998.292)
7005177,7245,363949,6429068,2024-12-01,0.695,0.0,72.5,8.048387,2.909677,5.864935,...,0.206065,0.000000,0.8,16.1,2024,Aucun,0.0,33,24729.333576,POINT (315997.597 1992998.292)
7005178,7245,363949,6429068,2024-05-01,0.770,0.0,94.9,15.264516,2.796774,8.711000,...,0.191290,0.000000,9.2,27.6,2024,Aucun,0.0,33,24729.333576,POINT (315997.597 1992998.292)


In [6]:
### Affichage de la liste des variables ###


list(gdf.columns)

['NUMERO',
 'LAMBX',
 'LAMBY',
 'DATE',
 'SWI_UNIF_MENS',
 'PRENEI',
 'PRELIQ',
 'T',
 'FF',
 'Q',
 'DLI',
 'SSI',
 'HU',
 'EVAP',
 'ETP',
 'PE',
 'SWI',
 'SSWI_10J',
 'DRAINC',
 'RUNC',
 'RESR_NEIGE',
 'RESR_NEIGE6',
 'HTEURNEIGE',
 'HTEURNEIGE6',
 'HTEURNEIGEX',
 'SNOW_FRAC',
 'ECOULEMENT',
 'WG_RACINE',
 'WGI_RACINE',
 'TINF_H',
 'TSUP_H',
 'YEAR',
 'ALEA',
 'NIVEAU',
 'DPT',
 'dist_to_argile',
 'geometry']

In [7]:
### Optimisation de la mémoire : conversion des float64 en float32 ###


# Sélection des colonnes de type float64
cols_float = gdf.select_dtypes(include=['float64']).columns

# Conversion en float32
gdf[cols_float] = gdf[cols_float].astype('float32')

gc.collect()

0

In [8]:
### Isolement de la variable de SWI uniforme mensuel (y) et des variables météorologiques ###


# Variable de SWI uniforme mensuel
y = gdf["SWI_UNIF_MENS"]

# Variables météorologiques
meteo_vars = [
    "T", "Q", "FF",
    "PRENEI", "PRELIQ",
    "SSI", "DLI", "ETP", "PE",
    "RESR_NEIGE", "RESR_NEIGE6",
    "HTEURNEIGE", "HTEURNEIGE6", "HTEURNEIGEX",
    "SNOW_FRAC", "ECOULEMENT",
    "TINF_H", "TSUP_H",
    "NIVEAU", "dist_to_argile"
]

In [9]:
### Dataframes des variables explicatives X et de la variable cible Y ###


# Liste des variables explicatives
X_cols = (
    meteo_vars +
    ["sin_month", "cos_month", "x", "y"]
)

In [12]:
gdf.geometry.x

0          723999.801614
1          715999.436117
2          708000.072246
3          699999.708485
4          667998.939984
               ...      
7005175    315997.596996
7005176    315997.596996
7005177    315997.596996
7005178    315997.596996
7005179    315997.596996
Length: 7005180, dtype: float64

In [13]:
gdf.geometry.y

0          2.280998e+06
1          2.280998e+06
2          2.280999e+06
3          2.280999e+06
4          2.176999e+06
               ...     
7005175    1.992998e+06
7005176    1.992998e+06
7005177    1.992998e+06
7005178    1.992998e+06
7005179    1.992998e+06
Length: 7005180, dtype: float64

In [ ]:
### Séparation des données en bases d'entraînement et de test ###


# Bases d'entraînement et de test
train = gdf[gdf["DATE"] < "2018-01-01"]
test  = gdf[gdf["DATE"] >= "2018-01-01"]

# X et y d'entraînement
X_train = train[X_cols]
y_train = train["SWI_UNIF_MENS"]

# X et y de test
X_test = test[X_cols]
y_test = test["SWI_UNIF_MENS"]

# Création de masques pour le découpage de la base d'entraînement
masque_train_train = gdf["DATE"] < "2017-01-01"
masque_val = (gdf["DATE"] >= "2017-01-01") & (gdf["DATE"] < "2018-01-01")

# Dates uniques de la base de test
dates_test = np.sort(gdf[gdf["DATE"] >= "2018-01-01"]['DATE'].unique())